In [0]:
from pyspark.sql.functions import *

CATALOG  = "olist_ecommerce"
GOLD     = f"{CATALOG}.gold"
SILVER   = f"{CATALOG}.silver"
BUSINESS = f"{CATALOG}.business"

# Verify tất cả tables cần thiết
print("Verifying source tables...")
sources = {
    f"{GOLD}.fact_orders":    "fact_orders",
    f"{GOLD}.dim_customer":   "dim_customer",
    f"{GOLD}.dim_product":    "dim_product",
    f"{GOLD}.dim_seller":     "dim_seller",
    f"{GOLD}.dim_date":       "dim_date",
    f"{SILVER}.reviews":      "reviews",
}

all_ok = True
for full_name, short_name in sources.items():
    try:
        count = spark.read.table(full_name).count()
        print(f"  ✅ {short_name}: {count:,} rows")
    except Exception as e:
        print(f"  ❌ {short_name}: {str(e)[:60]}")
        all_ok = False

if not all_ok:
    raise Exception("Some tables missing — check Gold layer!")

print("\n✅ All tables ready — starting Business views")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_revenue_daily AS

WITH daily_base AS (
    SELECT
        d.calendar_date,
        d.year,
        d.month,
        d.month_name,
        d.quarter,
        d.is_weekend,
        d.week_of_year,

        COUNT(DISTINCT f.order_id)      AS total_orders,
        COUNT(DISTINCT f.customer_key)  AS unique_customers,
        ROUND(SUM(f.payment_value), 2)  AS gross_revenue,
        ROUND(SUM(f.net_revenue), 2)    AS net_revenue,
        ROUND(SUM(f.freight_value), 2)  AS total_freight,
        ROUND(AVG(f.payment_value), 2)  AS avg_order_value,
        ROUND(AVG(f.lead_time_days), 1) AS avg_lead_days,
        ROUND(
            SUM(CASE WHEN f.payment_type = 'credit_card'
                THEN f.payment_value ELSE 0 END)
            / NULLIF(SUM(f.payment_value), 0) * 100, 1
        )                               AS credit_card_pct,
        ROUND(
            SUM(CASE WHEN f.is_delivered_on_time
                THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0), 1
        )                               AS on_time_rate_pct

    FROM {GOLD}.fact_orders f
    JOIN {GOLD}.dim_date d
        ON f.date_key = d.date_key
    WHERE f.order_status = 'delivered'
    GROUP BY
        d.calendar_date, d.year, d.month,
        d.month_name, d.quarter,
        d.is_weekend, d.week_of_year
),

daily_with_growth AS (
    SELECT
        *,
        -- 7-day rolling average
        ROUND(AVG(gross_revenue) OVER (
            ORDER BY calendar_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2)                           AS revenue_7d_avg,

        -- Day over Day growth %
        ROUND(
            (gross_revenue -
             LAG(gross_revenue, 1) OVER (ORDER BY calendar_date))
            / NULLIF(
                LAG(gross_revenue, 1) OVER (ORDER BY calendar_date),
            0) * 100, 1
        )                               AS dod_growth_pct,

        -- Week over Week growth %
        ROUND(
            (gross_revenue -
             LAG(gross_revenue, 7) OVER (ORDER BY calendar_date))
            / NULLIF(
                LAG(gross_revenue, 7) OVER (ORDER BY calendar_date),
            0) * 100, 1
        )                               AS wow_growth_pct,

        -- Cumulative revenue trong tháng
        ROUND(SUM(gross_revenue) OVER (
            PARTITION BY year, month
            ORDER BY calendar_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2)                           AS monthly_cumulative_revenue

    FROM daily_base
)

SELECT * FROM daily_with_growth
ORDER BY calendar_date
""")

print("✅ biz_revenue_daily")
spark.sql(f"""
    SELECT calendar_date, total_orders,
           gross_revenue, revenue_7d_avg,
           wow_growth_pct
    FROM {BUSINESS}.biz_revenue_daily
    ORDER BY gross_revenue DESC
    LIMIT 5
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_revenue_cohort AS

WITH customer_first_purchase AS (
    SELECT
        f.customer_key,
        MIN(d.year * 100 + d.month)  AS cohort_month,
        MIN(d.calendar_date)          AS first_purchase_date
    FROM {GOLD}.fact_orders f
    JOIN {GOLD}.dim_date d ON f.date_key = d.date_key
    WHERE f.order_status = 'delivered'
    GROUP BY f.customer_key
),

customer_orders AS (
    SELECT
        f.customer_key,
        f.order_id,
        f.payment_value,
        d.year * 100 + d.month          AS order_month,
        c.cohort_month,
        (d.year * 12 + d.month) -
        (c.cohort_month / 100 * 12 +
         MOD(c.cohort_month, 100))       AS months_since_cohort
    FROM {GOLD}.fact_orders f
    JOIN {GOLD}.dim_date d
        ON f.date_key = d.date_key
    JOIN customer_first_purchase c
        ON f.customer_key = c.customer_key
    WHERE f.order_status = 'delivered'
)

SELECT
    cohort_month,
    months_since_cohort,
    COUNT(DISTINCT customer_key)    AS active_customers,
    COUNT(DISTINCT order_id)        AS total_orders,
    ROUND(SUM(payment_value), 2)    AS cohort_revenue,
    ROUND(AVG(payment_value), 2)    AS avg_order_value
FROM customer_orders
GROUP BY cohort_month, months_since_cohort
ORDER BY cohort_month, months_since_cohort
""")

print("✅ biz_revenue_cohort")
spark.sql(f"""
    SELECT cohort_month, months_since_cohort,
           active_customers, cohort_revenue
    FROM {BUSINESS}.biz_revenue_cohort
    WHERE months_since_cohort <= 3
    ORDER BY cohort_month, months_since_cohort
    LIMIT 10
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_revenue_anomaly AS

WITH daily_stats AS (
    SELECT
        calendar_date,
        year,
        month,
        gross_revenue,
        total_orders,
        AVG(gross_revenue) OVER (
            ORDER BY calendar_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        )                               AS rolling_avg_7d,
        STDDEV(gross_revenue) OVER (
            ORDER BY calendar_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        )                               AS rolling_std_7d
    FROM {BUSINESS}.biz_revenue_daily
),

anomaly_scored AS (
    SELECT
        *,
        ROUND(
            (gross_revenue - rolling_avg_7d)
            / NULLIF(rolling_std_7d, 0),
        2)                              AS z_score,
        ROUND(
            (gross_revenue - rolling_avg_7d)
            / NULLIF(rolling_avg_7d, 0) * 100,
        1)                              AS pct_vs_avg
    FROM daily_stats
)

SELECT
    calendar_date,
    year,
    month,
    gross_revenue,
    total_orders,
    ROUND(rolling_avg_7d, 2)            AS rolling_avg_7d,
    z_score,
    pct_vs_avg,
    CASE
        WHEN z_score >  3.0 THEN 'CRITICAL_SPIKE'
        WHEN z_score >  2.0 THEN 'SPIKE'
        WHEN z_score < -3.0 THEN 'CRITICAL_DROP'
        WHEN z_score < -2.0 THEN 'DROP'
        ELSE                     'NORMAL'
    END                                 AS alert_type,
    CASE
        WHEN ABS(z_score) > 3.0 THEN 'HIGH'
        WHEN ABS(z_score) > 2.0 THEN 'MEDIUM'
        ELSE                          'LOW'
    END                                 AS severity
FROM anomaly_scored
ORDER BY calendar_date
""")

print("✅ biz_revenue_anomaly")
spark.sql(f"""
    SELECT calendar_date, gross_revenue,
           rolling_avg_7d, z_score,
           alert_type, severity
    FROM {BUSINESS}.biz_revenue_anomaly
    WHERE alert_type != 'NORMAL'
    ORDER BY ABS(z_score) DESC
    LIMIT 5
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_delivery_performance AS

WITH delivery_base AS (
    SELECT
        f.seller_state,
        f.customer_region,
        f.customer_state,
        f.delivery_speed_tier,

        COUNT(DISTINCT f.order_id)      AS total_orders,
        ROUND(AVG(f.lead_time_days), 1) AS avg_lead_days,
        ROUND(MIN(f.lead_time_days), 1) AS min_lead_days,
        ROUND(MAX(f.lead_time_days), 1) AS max_lead_days,
        ROUND(PERCENTILE_CONT(0.50)
            WITHIN GROUP (ORDER BY f.lead_time_days), 1
        )                               AS p50_lead_days,
        ROUND(PERCENTILE_CONT(0.90)
            WITHIN GROUP (ORDER BY f.lead_time_days), 1
        )                               AS p90_lead_days,
        ROUND(
            SUM(CASE WHEN f.is_delivered_on_time
                THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0), 1
        )                               AS on_time_rate_pct,
        ROUND(
            COUNT(CASE WHEN f.delivery_speed_tier
                = 'express' THEN 1 END) * 100.0
            / NULLIF(COUNT(*), 0), 1
        )                               AS express_pct,
        ROUND(
            COUNT(CASE WHEN f.delivery_speed_tier
                = 'very_slow' THEN 1 END) * 100.0
            / NULLIF(COUNT(*), 0), 1
        )                               AS very_slow_pct,
        ROUND(AVG(f.freight_value), 2)  AS avg_freight

    FROM {GOLD}.fact_orders f
    WHERE f.order_status = 'delivered'
      AND f.seller_state IS NOT NULL
    GROUP BY
        f.seller_state, f.customer_region,
        f.customer_state, f.delivery_speed_tier
)

SELECT
    *,
    ROUND(
        on_time_rate_pct * 0.5 +
        (100 - LEAST(avg_lead_days * 3, 100)) * 0.3 +
        (100 - very_slow_pct) * 0.2,
    1)                                  AS sla_health_score,
    CASE
        WHEN on_time_rate_pct >= 95
         AND avg_lead_days <= 7    THEN 'EXCELLENT'
        WHEN on_time_rate_pct >= 85
         AND avg_lead_days <= 14   THEN 'GOOD'
        WHEN on_time_rate_pct >= 70 THEN 'AVERAGE'
        ELSE                           'POOR'
    END                                 AS performance_tier
FROM delivery_base
ORDER BY total_orders DESC
""")

print("✅ biz_delivery_performance")
spark.sql(f"""
    SELECT seller_state, total_orders,
           avg_lead_days, on_time_rate_pct,
           sla_health_score, performance_tier
    FROM {BUSINESS}.biz_delivery_performance
    GROUP BY seller_state, total_orders, avg_lead_days,
             on_time_rate_pct, sla_health_score, performance_tier
    ORDER BY total_orders DESC
    LIMIT 5
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_seller_scorecard AS

WITH seller_metrics AS (
    SELECT
        s.seller_id,
        s.seller_state,
        s.brazil_region,

        COUNT(DISTINCT f.order_id)       AS total_orders,
        COUNT(DISTINCT f.customer_key)   AS unique_customers,
        ROUND(SUM(f.payment_value), 2)   AS total_revenue,
        ROUND(AVG(f.payment_value), 2)   AS avg_order_value,
        ROUND(AVG(f.lead_time_days), 1)  AS avg_lead_days,
        ROUND(
            SUM(CASE WHEN f.is_delivered_on_time
                THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0), 1
        )                                AS on_time_rate_pct,
        ROUND(AVG(r.review_score), 2)    AS avg_review_score,
        COUNT(r.review_id)               AS total_reviews,
        ROUND(
            SUM(CASE WHEN r.sentiment = 'negative'
                THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(r.review_id), 0), 1
        )                                AS negative_rate_pct,
        ROUND(AVG(r.response_time_hours), 1)
                                         AS avg_response_hours

    FROM {GOLD}.fact_orders f
    JOIN {GOLD}.dim_seller s
        ON f.seller_key = s.seller_key
    LEFT JOIN {SILVER}.reviews r
        ON f.order_id = r.order_id
    WHERE f.order_status = 'delivered'
    GROUP BY s.seller_id, s.seller_state, s.brazil_region
    HAVING COUNT(DISTINCT f.order_id) >= 10
),

seller_ranked AS (
    SELECT
        *,
        ROUND(
            (avg_review_score - 1) / 4.0 * 100 * 0.40 +
            on_time_rate_pct * 0.35 +
            (100 - LEAST(avg_lead_days * 4, 100)) * 0.25,
        1)                               AS seller_score,
        DENSE_RANK() OVER (
            ORDER BY total_revenue DESC
        )                                AS revenue_rank
    FROM seller_metrics
)

SELECT
    *,
    CASE
        WHEN seller_score >= 80 THEN 'STAR'
        WHEN seller_score >= 65 THEN 'GOOD'
        WHEN seller_score >= 50 THEN 'AVERAGE'
        ELSE                        'AT_RISK'
    END                                  AS seller_tier
FROM seller_ranked
ORDER BY seller_score DESC
""")

print("✅ biz_seller_scorecard")
spark.sql(f"""
    SELECT seller_id, seller_state,
           total_orders, total_revenue,
           avg_review_score, seller_score, seller_tier
    FROM {BUSINESS}.biz_seller_scorecard
    ORDER BY seller_score DESC
    LIMIT 5
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_product_quality AS

WITH category_metrics AS (
    SELECT
        p.product_category_name_english  AS category,
        p.size_tier,

        COUNT(DISTINCT f.order_id)       AS total_orders,
        ROUND(SUM(f.payment_value), 0)   AS total_revenue,
        ROUND(AVG(f.payment_value), 2)   AS avg_order_value,
        ROUND(AVG(f.items_price), 2)     AS avg_item_price,
        ROUND(AVG(f.freight_value), 2)   AS avg_freight,
        ROUND(AVG(f.lead_time_days), 1)  AS avg_lead_days,
        ROUND(AVG(r.review_score), 2)    AS avg_review_score,
        COUNT(r.review_id)               AS total_reviews,
        SUM(CASE WHEN r.sentiment = 'positive'
            THEN 1 ELSE 0 END)           AS positive_reviews,
        SUM(CASE WHEN r.sentiment = 'negative'
            THEN 1 ELSE 0 END)           AS negative_reviews,
        ROUND(
            SUM(CASE WHEN r.sentiment = 'negative'
                THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(r.review_id), 0), 1
        )                                AS negative_rate_pct,
        ROUND(
            AVG(f.freight_value)
            / NULLIF(AVG(f.items_price), 0) * 100,
        1)                               AS avg_freight_pct

    FROM {GOLD}.fact_orders f
    JOIN {GOLD}.dim_product p
        ON f.product_key = p.product_key
    LEFT JOIN {SILVER}.reviews r
        ON f.order_id = r.order_id
    WHERE f.order_status = 'delivered'
      AND p.product_category_name_english IS NOT NULL
    GROUP BY
        p.product_category_name_english,
        p.size_tier
    HAVING COUNT(DISTINCT f.order_id) >= 50
)

SELECT
    *,
    DENSE_RANK() OVER (
        ORDER BY total_revenue DESC
    )                                    AS revenue_rank,
    CASE
        WHEN avg_review_score >= 4.2
         AND negative_rate_pct <= 10  THEN 'HIGH_QUALITY'
        WHEN avg_review_score >= 3.8
         AND negative_rate_pct <= 20  THEN 'GOOD_QUALITY'
        WHEN avg_review_score >= 3.5  THEN 'AVERAGE_QUALITY'
        ELSE                              'LOW_QUALITY'
    END                                  AS quality_tier,
    CASE
        WHEN avg_lead_days > 20
         AND avg_review_score < 3.5   THEN TRUE
        ELSE                              FALSE
    END                                  AS slow_delivery_risk
FROM category_metrics
ORDER BY total_revenue DESC
""")

print("✅ biz_product_quality")
spark.sql(f"""
    SELECT category, total_orders, total_revenue,
           avg_review_score, negative_rate_pct,
           quality_tier
    FROM {BUSINESS}.biz_product_quality
    ORDER BY total_revenue DESC
    LIMIT 5
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_review_analysis AS

SELECT
    p.product_category_name_english  AS category,
    f.seller_state,
    f.customer_region,
    f.delivery_speed_tier,
    r.response_speed_tier,
    r.sentiment,
    d.year,
    d.month,

    COUNT(*)                         AS review_count,
    ROUND(AVG(r.review_score), 2)    AS avg_score,
    SUM(CASE WHEN r.has_comment
        THEN 1 ELSE 0 END)           AS with_comment,
    ROUND(AVG(r.response_time_hours), 1)
                                     AS avg_response_hours,
    ROUND(AVG(f.lead_time_days), 1)  AS avg_lead_days,

    -- On-time vs late delivery score comparison
    ROUND(AVG(CASE WHEN f.is_delivered_on_time
        THEN r.review_score END), 2) AS ontime_avg_score,
    ROUND(AVG(CASE WHEN NOT f.is_delivered_on_time
        THEN r.review_score END), 2) AS late_avg_score,
    ROUND(
        AVG(CASE WHEN f.is_delivered_on_time
            THEN r.review_score END) -
        AVG(CASE WHEN NOT f.is_delivered_on_time
            THEN r.review_score END),
    2)                               AS ontime_score_uplift

FROM {SILVER}.reviews r
JOIN {GOLD}.fact_orders f
    ON r.order_id = f.order_id
LEFT JOIN {GOLD}.dim_product p
    ON f.product_key = p.product_key
LEFT JOIN {GOLD}.dim_date d
    ON f.date_key = d.date_key
WHERE f.order_status = 'delivered'
GROUP BY
    p.product_category_name_english,
    f.seller_state, f.customer_region,
    f.delivery_speed_tier, r.response_speed_tier,
    r.sentiment, d.year, d.month
""")

print("✅ biz_review_analysis")
spark.sql(f"""
    SELECT delivery_speed_tier,
           SUM(review_count)              AS reviews,
           ROUND(AVG(avg_score), 2)       AS avg_score,
           ROUND(AVG(ontime_score_uplift), 2) AS score_uplift
    FROM {BUSINESS}.biz_review_analysis
    GROUP BY delivery_speed_tier
    ORDER BY avg_score DESC
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_customer_rfm AS

WITH customer_base AS (
    SELECT
        c.customer_unique_id,
        c.customer_state,
        c.brazil_region,
        MAX(d.calendar_date)             AS last_purchase_date,
        MIN(d.calendar_date)             AS first_purchase_date,
        COUNT(DISTINCT f.order_id)       AS frequency,
        ROUND(SUM(f.payment_value), 2)   AS monetary_value,
        ROUND(AVG(f.payment_value), 2)   AS avg_order_value,
        DATEDIFF(
            MAX(d.calendar_date),
            MIN(d.calendar_date)
        )                                AS tenure_days
    FROM {GOLD}.fact_orders f
    JOIN {GOLD}.dim_customer c
        ON f.customer_key = c.customer_key
    JOIN {GOLD}.dim_date d
        ON f.date_key = d.date_key
    WHERE f.order_status = 'delivered'
      AND c.is_current = true
    GROUP BY
        c.customer_unique_id,
        c.customer_state,
        c.brazil_region
),

rfm_scored AS (
    SELECT
        *,
        NTILE(5) OVER (
            ORDER BY last_purchase_date DESC
        )                                AS r_score,
        NTILE(5) OVER (
            ORDER BY frequency ASC
        )                                AS f_score,
        NTILE(5) OVER (
            ORDER BY monetary_value ASC
        )                                AS m_score
    FROM customer_base
)

SELECT
    *,
    ROUND((r_score + f_score + m_score) / 3.0, 1)
                                         AS rfm_score,
    CASE
        WHEN r_score >= 4
         AND f_score >= 4
         AND m_score >= 4   THEN 'Champions'
        WHEN r_score >= 3
         AND f_score >= 3   THEN 'Loyal Customers'
        WHEN r_score >= 4
         AND f_score <= 2   THEN 'New Customers'
        WHEN r_score >= 3
         AND m_score >= 4   THEN 'Potential Loyalists'
        WHEN r_score <= 2
         AND f_score >= 3
         AND m_score >= 3   THEN 'At Risk'
        WHEN r_score <= 2
         AND f_score >= 4   THEN 'Cannot Lose Them'
        WHEN r_score <= 2
         AND f_score <= 2   THEN 'Lost'
        ELSE                    'Needs Attention'
    END                                  AS rfm_segment
FROM rfm_scored
ORDER BY rfm_score DESC
""")

print("✅ biz_customer_rfm")
spark.sql(f"""
    SELECT rfm_segment,
           COUNT(*) AS customers,
           ROUND(AVG(monetary_value), 2) AS avg_ltv,
           ROUND(AVG(frequency), 1) AS avg_orders,
           ROUND(AVG(rfm_score), 1) AS avg_rfm_score
    FROM {BUSINESS}.biz_customer_rfm
    GROUP BY rfm_segment
    ORDER BY avg_ltv DESC
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_customer_ltv AS

WITH ltv_base AS (
    SELECT
        c.customer_unique_id,
        c.customer_state,
        c.brazil_region,
        f.payment_type                   AS preferred_payment,

        COUNT(DISTINCT f.order_id)       AS total_orders,
        ROUND(SUM(f.payment_value), 2)   AS lifetime_value,
        ROUND(AVG(f.payment_value), 2)   AS avg_order_value,
        ROUND(SUM(f.net_revenue), 2)     AS net_lifetime_value,
        MIN(d.calendar_date)             AS first_purchase,
        MAX(d.calendar_date)             AS last_purchase,
        DATEDIFF(
            MAX(d.calendar_date),
            MIN(d.calendar_date)
        )                                AS tenure_days,
        CASE
            WHEN COUNT(DISTINCT f.order_id) > 1
            THEN DATEDIFF(
                MAX(d.calendar_date),
                MIN(d.calendar_date)
            ) / (COUNT(DISTINCT f.order_id) - 1)
            ELSE NULL
        END                              AS avg_days_between_orders,
        COUNT(DISTINCT f.product_key)    AS unique_products,
        ROUND(AVG(f.lead_time_days), 1)  AS avg_lead_days,
        ROUND(
            SUM(CASE WHEN f.is_delivered_on_time
                THEN 1 ELSE 0 END) * 100.0
            / NULLIF(COUNT(*), 0), 1
        )                                AS on_time_rate_pct,
        ROUND(AVG(f.payment_installments), 1)
                                         AS avg_installments

    FROM {GOLD}.fact_orders f
    JOIN {GOLD}.dim_customer c
        ON f.customer_key = c.customer_key
    JOIN {GOLD}.dim_date d
        ON f.date_key = d.date_key
    WHERE f.order_status = 'delivered'
      AND c.is_current = true
    GROUP BY
        c.customer_unique_id,
        c.customer_state,
        c.brazil_region,
        f.payment_type
)

SELECT
    *,
    CASE
        WHEN lifetime_value >= 1000 THEN 'High Value'
        WHEN lifetime_value >= 300  THEN 'Mid Value'
        WHEN lifetime_value >= 100  THEN 'Low Value'
        ELSE                            'Micro Value'
    END                                  AS ltv_tier,
    CASE
        WHEN total_orders >= 5 THEN 'Loyal'
        WHEN total_orders >= 2 THEN 'Repeat'
        ELSE                       'One-Time'
    END                                  AS purchase_segment,
    -- Churn risk score (0=low, 100=high)
    ROUND(
        CASE
            WHEN avg_days_between_orders IS NULL
              OR avg_days_between_orders = 0 THEN 70.0
            ELSE LEAST(
                COALESCE(
                    try_divide(
                        DATEDIFF(DATE('2018-10-31'), last_purchase),
                        avg_days_between_orders
                    ), 0.0
                ) * 50.0, 100.0)
        END * 0.5 +
        CASE
            WHEN total_orders >= 5 THEN 0.0
            WHEN total_orders >= 3 THEN 30.0
            WHEN total_orders >= 2 THEN 60.0
            ELSE 90.0
        END * 0.3 +
        CASE
            WHEN on_time_rate_pct >= 90 THEN 0.0
            WHEN on_time_rate_pct >= 70 THEN 40.0
            ELSE 80.0
        END * 0.2,
    1)                                   AS churn_risk_score,
    DENSE_RANK() OVER (
        ORDER BY lifetime_value DESC
    )                                    AS revenue_rank
FROM ltv_base
ORDER BY lifetime_value DESC
""")

print("✅ biz_customer_ltv")
spark.sql(f"""
    SELECT ltv_tier, purchase_segment,
           COUNT(*) AS customers,
           ROUND(AVG(lifetime_value), 2) AS avg_ltv,
           ROUND(AVG(churn_risk_score), 1) AS avg_churn_risk
    FROM {BUSINESS}.biz_customer_ltv
    GROUP BY ltv_tier, purchase_segment
    ORDER BY avg_ltv DESC
""").show()

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {BUSINESS}.biz_customer_journey AS

WITH funnel_base AS (
    SELECT
        f.order_status,
        f.customer_region,
        d.year,
        d.month,
        COUNT(DISTINCT f.order_id)       AS orders,
        ROUND(SUM(f.payment_value), 0)   AS revenue,
        ROUND(AVG(f.lead_time_days), 1)  AS avg_lead_days,
        ROUND(AVG(f.approval_time_hours), 1)
                                         AS avg_approval_hours
    FROM {GOLD}.fact_orders f
    JOIN {GOLD}.dim_date d ON f.date_key = d.date_key
    GROUP BY
        f.order_status, f.customer_region,
        d.year, d.month
),

funnel_with_totals AS (
    SELECT
        *,
        SUM(orders) OVER (
            PARTITION BY year, month
        )                                AS total_orders_period
    FROM funnel_base
)

SELECT
    *,
    ROUND(
        orders * 100.0
        / NULLIF(total_orders_period, 0),
    1)                                   AS status_pct,
    CASE order_status
        WHEN 'delivered'   THEN 'Completed'
        WHEN 'shipped'     THEN 'In Transit'
        WHEN 'processing'  THEN 'Processing'
        WHEN 'approved'    THEN 'Approved'
        WHEN 'invoiced'    THEN 'Invoiced'
        WHEN 'canceled'    THEN 'Lost'
        WHEN 'unavailable' THEN 'Lost'
        ELSE                    'Other'
    END                                  AS funnel_stage,
    CASE order_status
        WHEN 'created'     THEN 1
        WHEN 'approved'    THEN 2
        WHEN 'processing'  THEN 3
        WHEN 'invoiced'    THEN 4
        WHEN 'shipped'     THEN 5
        WHEN 'delivered'   THEN 6
        WHEN 'canceled'    THEN 7
        WHEN 'unavailable' THEN 8
        ELSE                    9
    END                                  AS funnel_order
FROM funnel_with_totals
ORDER BY year, month, funnel_order
""")

print("✅ biz_customer_journey")
spark.sql(f"""
    SELECT order_status, funnel_stage,
           SUM(orders) AS total,
           ROUND(AVG(status_pct), 1) AS avg_pct
    FROM {BUSINESS}.biz_customer_journey
    GROUP BY order_status, funnel_stage
    ORDER BY total DESC
""").show()

In [0]:
print("=" * 60)
print("BUSINESS LAYER — FINAL VERIFICATION")
print("=" * 60)

views = {
    "Revenue & Growth": [
        "biz_revenue_daily",
        "biz_revenue_cohort",
        "biz_revenue_anomaly",
    ],
    "Operational Excellence": [
        "biz_delivery_performance",
        "biz_seller_scorecard",
        "biz_product_quality",
        "biz_review_analysis",
    ],
    "Customer Intelligence": [
        "biz_customer_rfm",
        "biz_customer_ltv",
        "biz_customer_journey",
    ],
}

total_ok = 0
for group, group_views in views.items():
    print(f"\n{group}:")
    for view in group_views:
        try:
            count = spark.read.table(
                f"{BUSINESS}.{view}").count()
            print(f"  ✅ {view}: {count:,} rows")
            total_ok += 1
        except Exception as e:
            print(f"  ❌ {view}: {str(e)[:50]}")

print(f"\n{'='*60}")
print(f"Views created: {total_ok}/10")
print(f"{'='*60}")

# Key insights
print("\n=== KEY INSIGHTS ===")

print("\n1. Top revenue anomalies (Black Friday?):")
spark.sql(f"""
    SELECT calendar_date, gross_revenue,
           z_score, alert_type
    FROM {BUSINESS}.biz_revenue_anomaly
    WHERE alert_type IN ('SPIKE','CRITICAL_SPIKE')
    ORDER BY z_score DESC
    LIMIT 3
""").show()

print("2. Customer RFM segments:")
spark.sql(f"""
    SELECT rfm_segment,
           COUNT(*) AS customers,
           ROUND(AVG(monetary_value), 0) AS avg_ltv
    FROM {BUSINESS}.biz_customer_rfm
    GROUP BY rfm_segment
    ORDER BY avg_ltv DESC
""").show()

print("3. Delivery quality by region:")
spark.sql(f"""
    SELECT customer_region,
           SUM(total_orders) AS orders,
           ROUND(AVG(on_time_rate_pct), 1) AS on_time_pct,
           ROUND(AVG(avg_lead_days), 1) AS avg_lead_days
    FROM {BUSINESS}.biz_delivery_performance
    GROUP BY customer_region
    ORDER BY orders DESC
""").show()

print("\n✅ Business layer COMPLETE — 10 views!")
print("\nNext steps:")
print("  1. Commit lên GitHub")
print("  2. Setup dbt + BigQuery")
print("  3. Looker Studio dashboard")
print("  4. Viết README.md")